<a href="https://colab.research.google.com/github/sweta96782/Assignment/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/sweta96782/Assignment.git

Cloning into 'Assignment'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 5.47 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
%cd Assignment

/content/Assignment


In [3]:
!ls -la

total 28
drwxr-xr-x 3 root root 4096 Nov 12 07:37 .
drwxr-xr-x 1 root root 4096 Nov 12 07:37 ..
drwxr-xr-x 8 root root 4096 Nov 12 07:37 .git
-rw-r--r-- 1 root root   24 Nov 12 07:37 README.md
-rw-r--r-- 1 root root 8020 Nov 12 07:37 sleep_pipeline.py
-rw-r--r-- 1 root root  906 Nov 12 07:37 Task_2.ipynb


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

project_path = "/content/drive/MyDrive/EEG_Assignment/task2"

os.makedirs(project_path, exist_ok=True)

print("Project folder:", project_path)

Project folder: /content/drive/MyDrive/EEG_Assignment/task2


In [9]:
!sed -n '1,100p' sleep_pipeline.py

"""
Sleep staging: CNN epoch-encoder + Transformer sequence model

Data   : Sleep-EDF Expanded, Sleep Cassette (SC) subset, fetched with MNE.
Task   : 5-class AASM staging (W, N1, N2, N3, REM). The model sees a window of
         SEQ_LEN consecutive 30-s epochs and predicts the stage of the CENTRE
         epoch of the window.
Metrics: accuracy, macro-F1 and Cohen's kappa on held-out data.

Run    : python sleep_pipeline.py
"""
import random

import mne
import numpy as np
import torch
import torch.nn as nn
from mne.datasets.sleep_physionet.age import fetch_data
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score)

SEED = 42
SUBJECTS = list(range(15))          # subjects 0-14, both nights where available
SEQ_LEN = 11                        # 11 x 30 s = 5.5 min of context
EPOCH_SEC = 30
CHANNELS = ["Fpz-Cz", "Pz-Oz", "horizontal"]   # names after infer_types=True
N_CLASSES = 5
CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"

In [10]:
!pip install mne tensorflow scikit-learn matplotlib pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 22.7 MB/s  0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mne]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [11]:
import mne
import tensorflow as tf
import sklearn
import numpy as np
import pandas as pd

print("MNE:", mne.__version__)
print("TensorFlow:", tf.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

MNE: 1.13.2
TensorFlow: 2.21.0
Scikit-learn: 1.6.1
NumPy: 2.4.6
Pandas: 2.2.3


In [12]:
os.makedirs("task2/original", exist_ok=True)
os.makedirs("task2/logs", exist_ok=True)

In [13]:
import shutil
shutil.copy(
    "sleep_pipeline.py",
    "task2/original/sleep_pipeline.py"
)

'task2/original/sleep_pipeline.py'

In [14]:
with open("task2/original/sleep_pipeline.py", "r") as f:
    print(f.read())

"""
Sleep staging: CNN epoch-encoder + Transformer sequence model

Data   : Sleep-EDF Expanded, Sleep Cassette (SC) subset, fetched with MNE.
Task   : 5-class AASM staging (W, N1, N2, N3, REM). The model sees a window of
         SEQ_LEN consecutive 30-s epochs and predicts the stage of the CENTRE
         epoch of the window.
Metrics: accuracy, macro-F1 and Cohen's kappa on held-out data.

Run    : python sleep_pipeline.py
"""
import random

import mne
import numpy as np
import torch
import torch.nn as nn
from mne.datasets.sleep_physionet.age import fetch_data
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score)

SEED = 42
SUBJECTS = list(range(15))          # subjects 0-14, both nights where available
SEQ_LEN = 11                        # 11 x 30 s = 5.5 min of context
EPOCH_SEC = 30
CHANNELS = ["Fpz-Cz", "Pz-Oz", "horizontal"]   # names after infer_types=True
N_CLASSES = 5
CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"

In [15]:
!python task2/original/sleep_pipeline.py

Using default location ~/mne_data for PHYSIONET_SLEEP...
Creating /root/mne_data
/content/Assignment/task2/original/sleep_pipeline.py:56: RuntimeWarning: Requested recording 2 for subject 13, but it is not available in corpus.
  return fetch_data(subjects=subjects, recording=[1, 2], on_missing="warn")
100%|██████████████████████████████████████| 48.3M/48.3M [00:00<00:00, 173GB/s]
100%|█████████████████████████████████████| 4.62k/4.62k [00:00<00:00, 24.9MB/s]
100%|██████████████████████████████████████| 51.6M/51.6M [00:00<00:00, 224GB/s]
100%|█████████████████████████████████████| 4.58k/4.58k [00:00<00:00, 26.4MB/s]
100%|██████████████████████████████████████| 51.1M/51.1M [00:00<00:00, 216GB/s]
100%|█████████████████████████████████████| 3.90k/3.90k [00:00<00:00, 23.9MB/s]
100%|██████████████████████████████████████| 52.0M/52.0M [00:00<00:00, 282GB/s]
100%|█████████████████████████████████████| 5.08k/5.08k [00:00<00:00, 31.9MB/s]
100%|██████████████████████████████████████| 51.1M/51.1M 

In [16]:
%%bash
set -o pipefail
python task2/original/sleep_pipeline.py 2>&1 | tee task2/logs/original_output.txt

Using default location ~/mne_data for PHYSIONET_SLEEP...
/content/Assignment/task2/original/sleep_pipeline.py:56: RuntimeWarning: Requested recording 2 for subject 13, but it is not available in corpus.
  return fetch_data(subjects=subjects, recording=[1, 2], on_missing="warn")
/content/Assignment/task2/original/sleep_pipeline.py:141: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(layer, num_layers=2)
train recordings: 24  test recordings: 5
train label counts: {'W': 45332, 'N1': 1902, 'N2': 10850, 'N3': 3354, 'REM': 4628}
epoch  0  loss 0.529  macro-F1 0.525
epoch  1  loss 0.623  macro-F1 0.572
epoch  2  loss 0.495  macro-F1 0.542
epoch  3  loss 0.870  macro-F1 0.586
epoch  4  loss 0.533  macro-F1 0.570
epoch  5  loss 0.812  macro-F1 0.572
epoch  6  loss 0.397  macro-F1 0.620
epoch  7  loss 1.103  macro-F1

In [17]:
!cat task2/logs/original_output.txt

Using default location ~/mne_data for PHYSIONET_SLEEP...
/content/Assignment/task2/original/sleep_pipeline.py:56: RuntimeWarning: Requested recording 2 for subject 13, but it is not available in corpus.
  return fetch_data(subjects=subjects, recording=[1, 2], on_missing="warn")
/content/Assignment/task2/original/sleep_pipeline.py:141: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(layer, num_layers=2)
train recordings: 24  test recordings: 5
train label counts: {'W': 45332, 'N1': 1902, 'N2': 10850, 'N3': 3354, 'REM': 4628}
epoch  0  loss 0.529  macro-F1 0.525
epoch  1  loss 0.623  macro-F1 0.572
epoch  2  loss 0.495  macro-F1 0.542
epoch  3  loss 0.870  macro-F1 0.586
epoch  4  loss 0.533  macro-F1 0.570
epoch  5  loss 0.812  macro-F1 0.572
epoch  6  loss 0.397  macro-F1 0.620
epoch  7  loss 1.103  macro-F1

In [18]:
!ls -lh task2/logs/

total 4.0K
-rw-r--r-- 1 root root 1.5K Nov 12 09:08 original_output.txt


In [22]:

!git config --global user.name "sweta96782"
!git config --global user.email "sweta.bhadra4@gmail.com"

In [23]:
!git add task2/original/sleep_pipeline.py task2/logs/original_output.txt
!git commit -m "Preserve original sleep staging pipeline and baseline output"

[main 18c9d4d] Preserve original sleep staging pipeline and baseline output
 2 files changed, 247 insertions(+)
 create mode 100644 task2/logs/original_output.txt
 create mode 100644 task2/original/sleep_pipeline.py


In [24]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address
